[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmcguir2/astromol/blob/refactor/docs/notebooks/02_reproduce_figures.ipynb)

# Reproduce census figures

This notebook is a pick-and-choose catalog of every figure currently supported by `astromol.figures`.

Run the setup and configuration cells first. After that, run only the figure cells you want. If you want all figures, use **Runtime -> Run all** and then run the final zip/download cell.


In [ ]:
# Run this setup cell first in Google Colab. In a local checkout with astromol
# already installed, it does nothing.
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/bmcguir2/astromol.git@refactor",
    ])


In [ ]:
from pathlib import Path
import shutil

from IPython.display import Image, display

from astromol.census import CensusView
from astromol.database import Database
from astromol.figures import (
    cumulative_by_atoms_data,
    cumulative_detection_data,
    detection_rate_by_atoms_data,
    du_by_source_type_data,
    du_histogram_data,
    facility_share_data,
    individual_source_data,
    kappa_histogram_data,
    mass_by_source_type_data,
    mass_by_wavelength_data,
    molecule_type_by_source_type_data,
    molecule_type_data,
    molecules_by_wavelength_atoms_data,
    periodic_heatmap_data,
    relative_du_by_source_type_data,
    rolling_rate_by_atoms_heatmap_data,
    scopes_by_year_data,
    source_type_data,
    wavelength_by_source_type_data,
    write_cumulative_by_atoms_plot,
    write_cumulative_detections_plot,
    write_detection_rate_by_atoms_comparison_plot,
    write_detection_rate_by_atoms_plot,
    write_du_bar_chart,
    write_du_by_source_type,
    write_du_by_source_type_boxplot,
    write_du_histogram,
    write_facility_share_bars_plot,
    write_facility_shares_plot,
    write_individual_source_pie_chart,
    write_kappa_histogram,
    write_mass_by_source_type,
    write_mass_by_source_type_boxplot,
    write_mass_by_wavelength_boxplot,
    write_mass_by_wavelength_plot,
    write_molecule_type_by_source_enrichment_matrix,
    write_molecule_type_by_source_type,
    write_molecules_by_wavelength_atoms_bubble_heatmap,
    write_molecules_by_wavelength_atoms_plot,
    write_periodic_heatmap,
    write_relative_du_by_source_type,
    write_relative_du_by_source_type_boxplot,
    write_rolling_rate_by_atoms_heatmap,
    write_scopes_by_year_plot,
    write_source_pie_chart,
    write_stacked_cumulative_by_atoms_plot,
    write_type_pie_chart,
    write_wavelength_by_source_type,
    write_wavelength_by_source_type_stacked_bar,
)


## Configuration

Choose the view and output format here.

- `VIEW_CHOICE = "2026"` reproduces the developing 2026 census view.
- `VIEW_CHOICE = "2021"` reproduces the 2021 census view where supported.
- `VIEW_CHOICE = "current"` uses the live database.
- `OUTPUT_FORMATS = ("png",)` makes quick preview files.
- `OUTPUT_FORMATS = ("pdf",)` makes manuscript-style files.
- `OUTPUT_FORMATS = ("png", "pdf")` makes both.


In [ ]:
VIEW_CHOICE = "2026"       # "2021", "2026", or "current"
OUTPUT_FORMATS = ("png",)  # choose ("png",), ("pdf",), or ("png", "pdf")
OUTPUT_DIR = Path("astromol_figure_outputs")
DISPLAY_PNG_PREVIEWS = True

OUTPUT_DIR.mkdir(exist_ok=True)

db = Database()

if VIEW_CHOICE == "current":
    view = CensusView.current(db)
    view_label = "current"
else:
    view = CensusView.for_census(db, VIEW_CHOICE)
    view_label = VIEW_CHOICE

view_2021 = CensusView.for_census(db, "2021")

print(f"Selected view: {view_label}")
print(f"Output formats: {OUTPUT_FORMATS}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
def write_figure(name, writer, data, **kwargs):
    """Write one figure in every selected output format."""
    paths = []
    for file_format in OUTPUT_FORMATS:
        output_path = OUTPUT_DIR / f"{name}.{file_format}"
        writer(data, output_path, **kwargs)
        paths.append(output_path)
        print(output_path)

    if DISPLAY_PNG_PREVIEWS:
        for output_path in paths:
            if output_path.suffix.lower() == ".png":
                display(Image(filename=str(output_path)))
                break
    return paths


## Cumulative detections

Secure, non-isotopologue ISM/CSM first detections through the selected view boundary.


In [ ]:
write_figure(
    "cumulative_detections",
    write_cumulative_detections_plot,
    cumulative_detection_data(view),
)


## Cumulative detections by atom-count category

Legacy-style line traces by atom-count category.


In [ ]:
by_atoms = cumulative_by_atoms_data(view)
write_figure(
    "cumulative_by_atoms",
    write_cumulative_by_atoms_plot,
    by_atoms,
)


## Stacked cumulative detections by atom-count category

Production companion view showing how atom-count categories contribute to the cumulative inventory.


In [ ]:
write_figure(
    "cumulative_by_atoms_stacked",
    write_stacked_cumulative_by_atoms_plot,
    by_atoms,
)


## Rolling detection-rate heatmap

Trailing rolling detections per year by atom-count category. The default window is 10 years.


In [ ]:
write_figure(
    "rolling_rate_by_atoms_heatmap",
    write_rolling_rate_by_atoms_heatmap,
    rolling_rate_by_atoms_heatmap_data(view, window=10),
)


## Periodic heatmap

Counts how many selected molecules contain each element.


In [ ]:
write_figure(
    "periodic_heatmap",
    write_periodic_heatmap,
    periodic_heatmap_data(view),
)


## Degree of unsaturation histogram

Legacy-style histogram for compatible molecular formulas. Fullerenes are excluded by default.


In [ ]:
du_data = du_histogram_data(view)
write_figure(
    "du_histogram_legacy",
    write_du_histogram,
    du_data,
)


## Degree of unsaturation bar chart

Production exact-value DU bar chart.


In [ ]:
write_figure(
    "du_bar_chart",
    write_du_bar_chart,
    du_data,
)


## Ray asymmetry parameter distribution

Kappa distribution for molecules with usable rotational constants. Linear molecules are included at kappa = -1.


In [ ]:
write_figure(
    "kappa_distribution",
    write_kappa_histogram,
    kappa_histogram_data(view),
)


## Molecule-type pie chart

Legacy categorical molecule-type summary.


In [ ]:
write_figure(
    "molecule_type_pie",
    write_type_pie_chart,
    molecule_type_data(view),
)


## Source-type pie chart

Legacy generalized first-detection source-type summary.


In [ ]:
write_figure(
    "source_type_pie",
    write_source_pie_chart,
    source_type_data(view),
)


## Individual-source pie chart

Legacy summary for major individual first-detection sources.


In [ ]:
write_figure(
    "individual_source_pie",
    write_individual_source_pie_chart,
    individual_source_data(view),
)


## Molecule type by source type

Legacy nested-pie visualization of molecule-type categories by generalized first-detection source type.


In [ ]:
type_source_data = molecule_type_by_source_type_data(view)
write_figure(
    "molecule_type_by_source_type_legacy",
    write_molecule_type_by_source_type,
    type_source_data,
)


## Molecule-type/source enrichment matrix

Production replacement for the nested-pie figure. Positive values indicate enrichment relative to the selected inventory average.


In [ ]:
write_figure(
    "molecule_type_source_enrichment",
    write_molecule_type_by_source_enrichment_matrix,
    type_source_data,
)


## Degree of unsaturation by source type

Legacy KDE source-category comparison. Fullerenes are excluded by default.


In [ ]:
du_source_data = du_by_source_type_data(view)
write_figure(
    "du_by_source_type_legacy",
    write_du_by_source_type,
    du_source_data,
)


## Degree of unsaturation by source type box/strip plot

Production source-category comparison for discrete DU values.


In [ ]:
write_figure(
    "du_by_source_type_boxplot",
    write_du_by_source_type_boxplot,
    du_source_data,
)


## Relative degree of unsaturation by source type

Legacy KDE comparison for relative DU values.


In [ ]:
relative_du_source_data = relative_du_by_source_type_data(view)
write_figure(
    "relative_du_by_source_type_legacy",
    write_relative_du_by_source_type,
    relative_du_source_data,
)


## Relative degree of unsaturation by source type box/strip plot

Production source-category comparison for relative DU values.


In [ ]:
write_figure(
    "relative_du_by_source_type_boxplot",
    write_relative_du_by_source_type_boxplot,
    relative_du_source_data,
)


## Molecular mass by source type

Legacy KDE source-category comparison. Fullerenes are excluded by default.


In [ ]:
mass_source_data = mass_by_source_type_data(view)
write_figure(
    "mass_by_source_type_legacy",
    write_mass_by_source_type,
    mass_source_data,
)


## Molecular mass by source type box/strip plot

Production source-category comparison for molecular masses.


In [ ]:
write_figure(
    "mass_by_source_type_boxplot",
    write_mass_by_source_type_boxplot,
    mass_source_data,
)


## Detection wavelength by source type

Legacy pie-grid visualization of first-detection wavelength contributions by generalized source type.


In [ ]:
wavelength_source_data = wavelength_by_source_type_data(view)
write_figure(
    "wavelength_by_source_type_legacy",
    write_wavelength_by_source_type,
    wavelength_source_data,
)


## Detection wavelength by source type stacked bar chart

Production replacement for the wavelength/source pie-grid.


In [ ]:
write_figure(
    "wavelength_by_source_type_stacked_bar",
    write_wavelength_by_source_type_stacked_bar,
    wavelength_source_data,
)


## Molecular mass by detection wavelength

Legacy KDE comparison by first-detection wavelength.


In [ ]:
mass_wave_data = mass_by_wavelength_data(view)
write_figure(
    "mass_by_wavelength_legacy",
    write_mass_by_wavelength_plot,
    mass_wave_data,
)


## Molecular mass by detection wavelength box/strip plot

Production wavelength comparison for molecular masses.


In [ ]:
write_figure(
    "mass_by_wavelength_boxplot",
    write_mass_by_wavelength_boxplot,
    mass_wave_data,
)


## Molecules by wavelength and atom count

Legacy KDE visualization of atom-count distributions by detection wavelength.


In [ ]:
wave_atoms_data = molecules_by_wavelength_atoms_data(view)
write_figure(
    "molecules_by_wavelength_atoms_legacy",
    write_molecules_by_wavelength_atoms_plot,
    wave_atoms_data,
)


## Molecules by wavelength and atom count bubble heatmap

Production replacement for the wavelength/atom-count KDE figure.


In [ ]:
write_figure(
    "molecules_by_wavelength_atoms_bubble_heatmap",
    write_molecules_by_wavelength_atoms_bubble_heatmap,
    wave_atoms_data,
)


## Detection rate by atom-count category

Average detections per year by category. This figure clips to the standard 2-12 atom domain plus PAHs and fullerenes.


In [ ]:
rate_data = detection_rate_by_atoms_data(view)
write_figure(
    "detection_rate_by_atoms",
    write_detection_rate_by_atoms_plot,
    rate_data,
)


## Detection rate by atom-count category with 2021 overlay

Production comparison plot using the selected view as the foreground and the 2021 census as the background reference.


In [ ]:
rate_2021 = detection_rate_by_atoms_data(view_2021)
for file_format in OUTPUT_FORMATS:
    output_path = OUTPUT_DIR / f"detection_rate_by_atoms_comparison.{file_format}"
    write_detection_rate_by_atoms_comparison_plot(
        rate_data,
        rate_2021,
        output_path,
        current_label=view_label,
        baseline_label="2021",
    )
    print(output_path)
    if DISPLAY_PNG_PREVIEWS and output_path.suffix.lower() == ".png":
        display(Image(filename=str(output_path)))


## Facility-share pie chart

Legacy facility-share chart.


In [ ]:
facility_data = facility_share_data(view)
write_figure(
    "facility_shares_legacy",
    write_facility_shares_plot,
    facility_data,
)


## Facility-share bar chart

Production replacement for the facility-share pie chart.


In [ ]:
write_figure(
    "facility_share_bars",
    write_facility_share_bars_plot,
    facility_data,
)


## Cumulative facility contributions by year

Modernized facility-contribution trace plot.


In [ ]:
write_figure(
    "scopes_by_year_modern",
    write_scopes_by_year_plot,
    scopes_by_year_data(view),
    style="modern",
)


## Cumulative facility contributions by year, legacy style

Legacy-style rendering of the same facility-contribution data.


In [ ]:
write_figure(
    "scopes_by_year_legacy",
    write_scopes_by_year_plot,
    scopes_by_year_data(view),
    style="legacy",
)


## Download generated figures

Run this cell after generating the figures you want. It creates a zip archive of the output directory. In Colab, it also prompts a download.


In [ ]:
archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"Created {archive_path}")

if IN_COLAB:
    from google.colab import files  # type: ignore
    files.download(archive_path)
else:
    print("Not running in Colab; archive is available at", archive_path)
